In [1]:
import pandas as pd
import pickle

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv("Dataset.csv")
print(df.shape, "\n")
print(df['target'].value_counts(normalize=True)*100, "\n")
print(df.isnull().sum()[df.isnull().sum()>0],"\n")

print(df.duplicated().sum(),"\n")
df = df.drop_duplicates()
print(df.duplicated().sum(),"\n")
print(df['target'].value_counts(normalize=True)*100, "\n")

df.columns = df.columns.str.strip().str.lower()

df.rename(columns={
    'age': 'Age',
    'sex': 'Gender',
    'chest pain type': 'ChestPainType',
    'resting bp s': 'RestingBP',
    'cholesterol': 'Cholesterol',
    'fasting blood sugar': 'FastingBS',
    'resting ecg': 'RestingECG',
    'max heart rate': 'MaxHR',
    'exercise angina': 'ExerciseAngina',
    'oldpeak': 'ST_Depression',
    'st slope': 'ST_Slope',
    'target': 'HeartDisease'
}, inplace=True)

df.info()
print("\n")

x = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

categorical_cols = [
    'ChestPainType','RestingECG','ST_Slope'
]

numerical_cols = [
    'Age','Cholesterol','RestingBP','MaxHR','ST_Depression'
]

x_encoded = pd.get_dummies(x, columns=categorical_cols, drop_first=True)
x_encoded = x_encoded.astype(int)

print(x_encoded.dtypes, "\n")

print(x_encoded.select_dtypes(include=['object']).columns, "\n")

print(x.shape)
print("Final Training Feature Shape:", x_encoded.shape)

x_train, x_test, y_train, y_test = train_test_split(
    x_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(y_train.value_counts(normalize=True)*100, "\n")

scaler = StandardScaler()

x_train[numerical_cols] = scaler.fit_transform(x_train[numerical_cols])
x_test[numerical_cols] = scaler.transform(x_test[numerical_cols])


lr = LogisticRegression()
lr.fit(x_train, y_train)
lr_pred = lr.predict(x_test)

rf = RandomForestClassifier()
rf.fit(x_train, y_train)
rf_pred = rf.predict(x_test)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(x_train, y_train)
xgb_pred = xgb.predict(x_test)

print("Logistic Regression:\n")
print(classification_report(y_test, lr_pred),"\n")

print("Accuracy:", accuracy_score(y_test, lr_pred),"\n")

print("Random Forest Classifier:\n")
print(classification_report(y_test, rf_pred),"\n")

print("Accuracy:", accuracy_score(y_test, rf_pred),"\n")

print("XGBoost:\n")
print(classification_report(y_test, xgb_pred))

print("XGB Accuracy:", accuracy_score(y_test, xgb_pred))

pickle.dump(lr, open("heart_disease_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl","wb"))

print("Model Trained Successfully! You can use it now...")


(1190, 12) 

target
1    52.857143
0    47.142857
Name: proportion, dtype: float64 

Series([], dtype: int64) 

272 

0 

target
1    55.337691
0    44.662309
Name: proportion, dtype: float64 

<class 'pandas.core.frame.DataFrame'>
Index: 918 entries, 0 to 1189
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Gender          918 non-null    int64  
 2   ChestPainType   918 non-null    int64  
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    int64  
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    int64  
 9   ST_Depression   918 non-null    float64
 10  ST_Slope        918 non-null    int64  
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(11)
memory usage: 93.2 KB


Age                in

c:\Users\Sachin Chaudhary\OneDrive\Desktop\heartNew\myenv\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:22:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
